<a href="https://colab.research.google.com/github/thomasdukullay/Duks-/blob/main/the_GameStop_Stock_Data_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install yfinance beautifulsoup4 lxml html5lib plotly pandas requests -q
import yfinance as yf, pandas as pd, requests
from bs4 import BeautifulSoup
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [3]:
tesla = yf.Ticker("TSLA")
tesla_data = tesla.history(period="max")
tesla_data.reset_index(inplace=True)
tesla_data.head()

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,2010-06-29 00:00:00-04:00,1.266667,1.666667,1.169333,1.592667,281494500,0.0,0.0
1,2010-06-30 00:00:00-04:00,1.719333,2.028000,1.553333,1.588667,257806500,0.0,0.0
2,2010-07-01 00:00:00-04:00,1.666667,1.728000,1.351333,1.464000,123282000,0.0,0.0
3,2010-07-02 00:00:00-04:00,1.533333,1.540000,1.247333,1.280000,77097000,0.0,0.0
4,2010-07-06 00:00:00-04:00,1.333333,1.333333,1.055333,1.074000,103003500,0.0,0.0


In [4]:
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/revenue.htm"
html_data = requests.get(url).text
soup = BeautifulSoup(html_data, "html.parser")
rows = []
for table in soup.find_all("table"):
    if "Tesla Quarterly Revenue" in table.get_text():
        for tr in table.find("tbody").find_all("tr"):
            td = tr.find_all("td")
            if len(td) == 2:
                rows.append({"Date": td[0].text.strip(), "Revenue": td[1].text.strip()})
        break
tesla_revenue = pd.DataFrame(rows)
tesla_revenue["Revenue"] = tesla_revenue["Revenue"].str.replace(r"[$,]", "", regex=True)
tesla_revenue = tesla_revenue[tesla_revenue["Revenue"] != ""].dropna()
tesla_revenue.tail()

,Date,Revenue
48,2010-09-30,31
49,2010-06-30,28
50,2010-03-31,21
52,2009-09-30,46
53,2009-06-30,27


In [5]:
gme = yf.Ticker("GME")
gme_data = gme.history(period="max")
gme_data.reset_index(inplace=True)
gme_data.head()

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,2002-02-13 00:00:00-05:00,1.620128,1.693350,1.603296,1.691666,76216000,0.0,0.0
1,2002-02-14 00:00:00-05:00,1.712707,1.716074,1.670626,1.683250,11021600,0.0,0.0
2,2002-02-15 00:00:00-05:00,1.683250,1.687458,1.658001,1.674834,8389600,0.0,0.0
3,2002-02-19 00:00:00-05:00,1.666417,1.666417,1.578047,1.607504,7410400,0.0,0.0
4,2002-02-20 00:00:00-05:00,1.615920,1.662210,1.603296,1.662210,6892800,0.0,0.0


In [6]:
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/stock.html"
html_data_2 = requests.get(url).text
soup = BeautifulSoup(html_data_2, "html.parser")
rows = []
for table in soup.find_all("table"):
    if "GameStop Quarterly Revenue" in table.get_text():
        for tr in table.find("tbody").find_all("tr"):
            td = tr.find_all("td")
            if len(td) == 2:
                rows.append({"Date": td[0].text.strip(), "Revenue": td[1].text.strip()})
        break
gme_revenue = pd.DataFrame(rows)
gme_revenue["Revenue"] = gme_revenue["Revenue"].str.replace(r"[$,]", "", regex=True)
gme_revenue = gme_revenue[gme_revenue["Revenue"] != ""].dropna()
gme_revenue.tail()

,Date,Revenue
57,2006-01-31,1667
58,2005-10-31,534
59,2005-07-31,416
60,2005-04-30,475
61,2005-01-31,709


In [11]:
def make_graph(stock_data, revenue_data, stock):
    stock_data = stock_data[stock_data['Date'] <= '2021-06-14']
    revenue_data = revenue_data[revenue_data['Date'] <= '2021-04-30']
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
        subplot_titles=("Historical Share Price", "Historical Revenue"), vertical_spacing=.3)
    fig.add_trace(go.Scatter(x=pd.to_datetime(stock_data['Date']), y=stock_data['Close'].astype('float')), row=1, col=1)
    fig.add_trace(go.Scatter(x=pd.to_datetime(revenue_data['Date']), y=revenue_data['Revenue'].astype('float')), row=2, col=1)
    fig.update_layout(showlegend=False, height=900, title=stock)
    fig.show()

In [12]:
make_graph(tesla_data, tesla_revenue, "Tesla")

In [13]:
make_graph(gme_data, gme_revenue, "GameStop")